# Notebook 07 — Model Evaluation & Winner Selection
**CIS6035 IHMS — Somerset Mirissa Beach Hotel**

Evaluates all three models on the **2025 test set** (hold-out, never seen during training or tuning).

Outputs:
- Comparison table (RMSE, MAE, MAPE, training time)
- Overlay plot: actual vs all 3 models on 2025
- Winner selection (lowest RMSE)
- `models/model_metadata.json`
- `docs/model_comparison.md`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import json
import time
import warnings
import os
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX

warnings.filterwarnings('ignore')
os.makedirs('../models', exist_ok=True)
os.makedirs('../docs', exist_ok=True)

plt.rcParams['figure.figsize'] = (16, 6)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((np.array(y_true) - np.array(y_pred))**2))
def mae(y_true, y_pred):
    return np.mean(np.abs(np.array(y_true) - np.array(y_pred)))
def mape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

In [ ]:
# Load all splits (for context) and the test set
train = pd.read_csv('../data/splits/train.csv', index_col='date', parse_dates=True)['occupancy_rate']
val   = pd.read_csv('../data/splits/val.csv',   index_col='date', parse_dates=True)['occupancy_rate']
test  = pd.read_csv('../data/splits/test.csv',  index_col='date', parse_dates=True)['occupancy_rate']

# Full history up to end of val for re-fitting before test
train_val = pd.concat([train, val])
print(f'Test set: {len(test)} days ({test.index.min().date()} → {test.index.max().date()})')

In [ ]:
# Load artifacts
arima_art  = joblib.load('../models/arima_model.pkl')
sarima_art = joblib.load('../models/sarima_model.pkl')
prophet_art = joblib.load('../models/prophet_model.pkl')

print('Artifacts loaded.')
print(f'ARIMA order: {arima_art["order"]}')
print(f'SARIMA seasonal: {sarima_art["seasonal_order"]}')
print(f'Prophet cps: {prophet_art["changepoint_prior_scale"]}')

In [ ]:
# === ARIMA on test ===
print('Evaluating ARIMA on test set...')
t0 = time.time()
history = list(train_val)
arima_preds = []
for actual in test:
    m = ARIMA(history, order=arima_art['order']).fit(method_kwargs={'warn_convergence': False})
    pred = float(np.clip(m.forecast(1)[0], 0, 1))
    arima_preds.append(pred)
    history.append(actual)
arima_time = time.time() - t0

arima_rmse = rmse(test, arima_preds)
arima_mae  = mae(test, arima_preds)
arima_mape = mape(test, arima_preds)
print(f'ARIMA — RMSE: {arima_rmse:.4f}  MAE: {arima_mae:.4f}  MAPE: {arima_mape:.2f}%')

In [ ]:
# === SARIMA on test ===
print('Evaluating SARIMA on test set...')
t0 = time.time()
history = list(train_val)
sarima_preds = []
for actual in test:
    m = SARIMAX(
        history,
        order=sarima_art['order'],
        seasonal_order=sarima_art['seasonal_order'],
        enforce_stationarity=False,
        enforce_invertibility=False,
    ).fit(disp=False)
    pred = float(np.clip(m.forecast(1)[0], 0, 1))
    sarima_preds.append(pred)
    history.append(actual)
sarima_time = time.time() - t0

sarima_rmse = rmse(test, sarima_preds)
sarima_mae  = mae(test, sarima_preds)
sarima_mape = mape(test, sarima_preds)
print(f'SARIMA — RMSE: {sarima_rmse:.4f}  MAE: {sarima_mae:.4f}  MAPE: {sarima_mape:.2f}%')

In [ ]:
# === Prophet on test ===
print('Evaluating Prophet on test set...')
t0 = time.time()
prophet_model = prophet_art['model']
future = prophet_model.make_future_dataframe(periods=len(test) + len(val), freq='D')
forecast = prophet_model.predict(future)
test_forecast = forecast.tail(len(test))
prophet_preds = np.clip(test_forecast['yhat'].values, 0, 1)
prophet_lower = np.clip(test_forecast['yhat_lower'].values, 0, 1)
prophet_upper = np.clip(test_forecast['yhat_upper'].values, 0, 1)
prophet_time = time.time() - t0

prophet_rmse = rmse(test, prophet_preds)
prophet_mae  = mae(test, prophet_preds)
prophet_mape = mape(test, prophet_preds)
print(f'Prophet — RMSE: {prophet_rmse:.4f}  MAE: {prophet_mae:.4f}  MAPE: {prophet_mape:.2f}%')

In [ ]:
# === Comparison Table ===
comparison = pd.DataFrame([
    {'Model': f'ARIMA{arima_art["order"]}', 'RMSE': arima_rmse, 'MAE': arima_mae, 'MAPE (%)': arima_mape, 'Time (s)': round(arima_time, 1)},
    {'Model': f'SARIMA',                    'RMSE': sarima_rmse, 'MAE': sarima_mae, 'MAPE (%)': sarima_mape, 'Time (s)': round(sarima_time, 1)},
    {'Model': 'Prophet',                     'RMSE': prophet_rmse, 'MAE': prophet_mae, 'MAPE (%)': prophet_mape, 'Time (s)': round(prophet_time, 1)},
])
comparison = comparison.set_index('Model')
comparison = comparison.round(4)

print('\n=== Model Comparison on Test Set (2025) ===')
print(comparison.to_string())

In [ ]:
# Overlay plot
plt.figure(figsize=(16, 6))
plt.plot(test.index, test.values, label='Actual', color='#0f172a', linewidth=1.5)
plt.plot(test.index, arima_preds,  label=f'ARIMA (RMSE={arima_rmse:.4f})',  color='#ef4444', linewidth=1.2, linestyle='--')
plt.plot(test.index, sarima_preds, label=f'SARIMA (RMSE={sarima_rmse:.4f})', color='#8b5cf6', linewidth=1.2, linestyle=':')
plt.plot(test.index, prophet_preds,label=f'Prophet (RMSE={prophet_rmse:.4f})',color='#f59e0b', linewidth=1.5, linestyle='-.')
plt.fill_between(test.index, prophet_lower, prophet_upper, color='#fde68a', alpha=0.3, label='Prophet 90% CI')
plt.title('Model Comparison — Test Set 2025', fontsize=14, fontweight='bold')
plt.ylabel('Occupancy Rate')
plt.legend()
plt.tight_layout()
plt.savefig('../docs/figures/07_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Winner selection: lowest RMSE on test
all_rmse = {'arima': arima_rmse, 'sarima': sarima_rmse, 'prophet': prophet_rmse}
winner = min(all_rmse, key=all_rmse.get)
winner_rmse = all_rmse[winner]

print(f'\n>>> WINNER: {winner.upper()} (Test RMSE = {winner_rmse:.4f})')

# Save model_metadata.json
metadata = {
    'winner': winner,
    'test_rmse': round(winner_rmse, 4),
    'test_mae': round(all_rmse.get(winner + '_mae', 0), 4),
    'all_models': {
        'arima':   {'rmse': round(arima_rmse, 4),  'mae': round(arima_mae, 4),  'mape': round(arima_mape, 2)},
        'sarima':  {'rmse': round(sarima_rmse, 4), 'mae': round(sarima_mae, 4), 'mape': round(sarima_mape, 2)},
        'prophet': {'rmse': round(prophet_rmse, 4),'mae': round(prophet_mae, 4),'mape': round(prophet_mape, 2)},
    },
    'trained_on': str(train_val.index.max().date()),
    'evaluated_on': str(test.index.max().date()),
}

with open('../models/model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print('Saved models/model_metadata.json')
print(json.dumps(metadata, indent=2))

In [ ]:
# Copy the winning model artifact to best_model.pkl
import shutil
winner_pkl = f'../models/{winner}_model.pkl'
shutil.copy(winner_pkl, '../models/best_model.pkl')
print(f'Copied {winner_pkl} → models/best_model.pkl')

In [ ]:
# Write docs/model_comparison.md
md_content = f"""# Model Comparison Report

**Project:** CIS6035 IHMS — Somerset Mirissa Beach Hotel

## Methodology

Three univariate time series models were trained and evaluated for 30-day occupancy forecasting:

| Split | Period | Days |
|-------|--------|------|
| Train | 2023-01-01 → 2023-12-31 | 365 |
| Validation | 2024-01-01 → 2024-12-31 | 366 |
| **Test (held-out)** | **2025-01-01 → 2025-12-31** | **365** |

## Results on Test Set (2025)

{comparison.to_markdown()}

## Winner

**{winner.upper()}** achieves the lowest RMSE ({winner_rmse:.4f}) on the 2025 test set and is deployed in production.

## Model Details

### ARIMA
- Order: {arima_art['order']}, determined via AIC grid search
- Univariate model, no seasonality term
- Strength: simple, interpretable
- Weakness: cannot capture strong seasonal patterns without differencing

### SARIMA
- Seasonal period s=7 (weekly)
- Seasonal order: {sarima_art['seasonal_order']}
- Better than ARIMA at capturing weekly cycles
- Higher computational cost for rolling forecasts

### Prophet
- Framework by Meta for time series with strong seasonality
- Includes Sri Lanka public holiday effects
- Custom quarterly seasonality
- Best at capturing yearly and holiday patterns
- Provides uncertainty intervals (90% CI)

## Conclusion

{winner.capitalize()} is selected as the production model due to its superior RMSE on unseen 2025 data.
It handles the strong seasonal patterns in Sri Lanka beach tourism (peak Dec-Jan, secondary Apr) better
than the baseline ARIMA approach.
"""

os.makedirs('../docs', exist_ok=True)
with open('../docs/model_comparison.md', 'w') as f:
    f.write(md_content)

print('Saved docs/model_comparison.md')